# 🚀 OGBN-Arxiv Graph Neural Network Classification: End-to-End Pipeline
<a href="https://colab.research.google.com/github/sankalpams/OGBN-Arxiv-GNN-Classification/blob/main/notebooks/all_in_one_ogbn_arxiv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This comprehensive all-in-one notebook implements the entire machine learning pipeline on the **`ogbn-arxiv`** citation network:
1. **Environment Setup & PyTorch Tensor Fundamentals** (Task 01)
2. **Graph Representation & Exploratory Data Analysis (EDA)** (Task 02)
3. **Data Preparation & Temporal Splits** (Task 03)
4. **Graph Convolutional Network (GCN) Baseline** (Task 04)
5. **Graph Attention Network (GAT) Architecture** (Task 05)
6. **Hyperparameter Tuning & Optimisation** (Task 06)
7. **Comprehensive Evaluation & Benchmark Comparison** (Task 07)
8. **Model Explainability, Embeddings & Homophily Analysis** (Task 08)
9. **Interactive Paper Category Inference Demo**

In [ ]:
# ==============================================================================
# Setup Environment (Google Colab & Local Jupyter Compatibility)
# ==============================================================================
import os
import sys
from pathlib import Path

# Detect Google Colab environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab. Installing dependencies...")
    !pip install -q torch-geometric ogb
    
    REPO_NAME = 'OGBN-Arxiv-GNN-Classification'
    REPO_URL = f'https://github.com/sankalpams/{REPO_NAME}.git'
    
    if not os.path.exists(REPO_NAME) and not Path('src').exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL}
        
    REPO_DIR = Path(f'/content/{REPO_NAME}')
    if REPO_DIR.exists():
        %cd {REPO_DIR}
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        PROJECT_ROOT = REPO_DIR
    else:
        PROJECT_ROOT = Path().resolve()
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
else:
    print("💻 Running locally.")
    PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Active Project Root: {PROJECT_ROOT}")

# Global Imports and Seed Setting
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for deterministic reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ Device allocated: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")

---
## 1. PyTorch & Tensor Fundamentals (Task 01)
Demonstrates multi-dimensional tensor creation, memory layout, linear algebra operations, broadcasting, and GPU memory migration.

In [ ]:
# Create 2D float tensor
tensor = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("Original 3x4 Tensor:\n", tensor)
print("Row slicing (index 1):", tensor[1])
print("Reshaped to 2x6:\n", tensor.reshape(2, 6))

# Matrix multiplication
weights = torch.randn(4, 2)
print("Matrix product (3x4 @ 4x2 -> 3x2):\n", tensor @ weights)

# Broadcasting addition
bias = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("Broadcasting addition:\n", tensor + bias)

# Reduction / Aggregations
print("Column-wise means:", tensor.mean(dim=0))

# GPU allocation
gpu_tensor = tensor.to(device)
print(f"Tensor migrated to: {gpu_tensor.device}")

---
## 2. Graph Representation & Exploratory Analysis (Task 02)
Load the official Open Graph Benchmark `ogbn-arxiv` citation network (169,343 papers, 1,166,243 directed citation edges, 40 computer science categories).

In [ ]:
from src.config import RAW_DATA_DIR, RESULTS_DIR
from src.data import load_ogbn_arxiv
from src.graph.graph_analysis import graph_summary
from src.graph.visualization import plot_sample_subgraph

dataset, data, splits = load_ogbn_arxiv(RAW_DATA_DIR)
summary = graph_summary(data)

summary_df = pd.DataFrame([summary])
print("=== Graph Summary Metrics ===")
display(summary_df)

out_graph = RESULTS_DIR / 'graph_analysis'
out_graph.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_graph / 'graph_summary.csv', index=False)
(out_graph / 'graph_density.txt').write_text(f"Graph density: {summary['density']:.8f}\n")

# Degree distribution plot
degrees = data.edge_index[0].bincount(minlength=data.num_nodes).cpu().numpy()
plt.figure(figsize=(9, 4))
plt.hist(degrees, bins=100, log=True, color='royalblue', edgecolor='black', alpha=0.8)
plt.xlabel('Degree (Number of outgoing citations)')
plt.ylabel('Node count (Log Scale)')
plt.title('Node Out-Degree Distribution in ogbn-arxiv')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(out_graph / 'degree_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

# Sample ego-subgraph visualization
plot_sample_subgraph(data, out_graph / 'sample_subgraph.png')

---
## 3. Data Preparation & Temporal Splits (Task 03)
Verify official temporal split indexing to avoid data leakage across publication years.

In [ ]:
from src.config import PROCESSED_DATA_DIR
from src.data import prepare_and_save_data
from src.data.splits import split_summary

prepared_data = prepare_and_save_data(data, splits, PROCESSED_DATA_DIR, normalize=False)
split_stats = split_summary(splits)

print(f"Node feature tensor shape: {prepared_data.x.shape} (128-d word embeddings)")
print(f"Target label tensor shape: {prepared_data.y.shape} ({dataset.num_classes} distinct CS categories)")
print(f"Temporal Split Distribution: {split_stats}")

# Train/Val/Test Split Pie Chart
split_counts = [len(splits['train']), len(splits['valid']), len(splits['test'])]
plt.figure(figsize=(6, 6))
plt.pie(split_counts, labels=['Train (<=2017)', 'Val (2018)', 'Test (2019-2020)'], 
        autopct='%1.1f%%', colors=['#4CAF50', '#FF9800', '#2196F3'], startangle=140)
plt.title('ogbn-arxiv Official Temporal Splits')
plt.show()

---
## 4. Graph Convolutional Network (GCN) Baseline (Task 04)
Train a 2-layer GCN model with ReLU activation, Dropout, and hidden representation extraction.

In [ ]:
from src.config import MODELS_DIR
from src.models import GCN
from src.training import fit

data_dev = data.to(device)
split_dev = {k: v.to(device) for k, v in splits.items()}

MODELS_DIR.mkdir(parents=True, exist_ok=True)
set_seed(42)

gcn_model = GCN(data.num_features, 256, dataset.num_classes, dropout=0.5).to(device)
print(gcn_model)

gcn_history = fit(
    gcn_model, 
    data_dev, 
    split_dev, 
    epochs=30, 
    learning_rate=0.01, 
    checkpoint_path=MODELS_DIR / 'best_gcn.pt'
)

out_train = RESULTS_DIR / 'training'
out_train.mkdir(parents=True, exist_ok=True)
gcn_history.to_csv(out_train / 'gcn_training_history.csv', index=False)

# Plot GCN Training Loss & Validation Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(gcn_history['epoch'], gcn_history['loss'], 'b-', lw=2, label='Training Loss')
ax1.set_title('GCN Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

ax2.plot(gcn_history['epoch'], gcn_history['validation_accuracy'] * 100, 'g-', lw=2, label='Val Accuracy')
ax2.set_title('GCN Validation Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()
plt.show()

---
## 5. Graph Attention Network (GAT) Model (Task 05)
Train a multi-head Graph Attention Network with 4 attention heads and self-attention mechanism.

In [ ]:
from src.models import GAT

set_seed(42)
gat_model = GAT(data.num_features, 64, dataset.num_classes, heads=4, dropout=0.5).to(device)
print(gat_model)

gat_history = fit(
    gat_model, 
    data_dev, 
    split_dev, 
    epochs=30, 
    learning_rate=0.005, 
    checkpoint_path=MODELS_DIR / 'best_gat.pt'
)

gat_history.to_csv(out_train / 'gat_training_history.csv', index=False)

# Plot GAT Training Loss & Validation Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(gat_history['epoch'], gat_history['loss'], 'r-', lw=2, label='Training Loss')
ax1.set_title('GAT Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

ax2.plot(gat_history['epoch'], gat_history['validation_accuracy'] * 100, 'm-', lw=2, label='Val Accuracy')
ax2.set_title('GAT Validation Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()
plt.show()

---
## 6. Hyperparameter Tuning & Optimisation (Task 06)
Systematic grid search over hidden channels and dropout rates.

In [ ]:
from src.training.hyperparameter_tuning import parameter_grid

trials = parameter_grid({
    'hidden_channels': [128, 256], 
    'dropout': [0.3, 0.5], 
    'learning_rate': [0.01]
})

print(f"Running {len(trials)} hyperparameter trials...")
results = []
for i, params in enumerate(trials, 1):
    set_seed(42)
    model = GCN(data.num_features, params['hidden_channels'], dataset.num_classes, params['dropout']).to(device)
    hist = fit(model, data_dev, split_dev, epochs=20, learning_rate=params['learning_rate'])
    best_val_acc = hist['validation_accuracy'].max()
    print(f"Trial {i}/{len(trials)}: {params} -> Peak Val Acc = {best_val_acc*100:.2f}%")
    results.append({**params, 'validation_accuracy': best_val_acc})

tuning_df = pd.DataFrame(results).sort_values('validation_accuracy', ascending=False)
tuning_df.to_csv(out_train / 'hyperparameter_trials.csv', index=False)
display(tuning_df)

---
## 7. Comprehensive Model Evaluation & Comparison (Task 07)
Evaluate GCN and GAT on the unseen official test set using Accuracy, Weighted Precision, Recall, and F1-score.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from src.evaluation import evaluate_model
from src.evaluation.comparison import plot_model_comparison

eval_models = {
    'GCN': GCN(data.num_features, 256, dataset.num_classes).to(device),
    'GAT': GAT(data.num_features, 64, dataset.num_classes, heads=4).to(device)
}

out_eval = RESULTS_DIR / 'evaluation'
out_eval.mkdir(parents=True, exist_ok=True)

rows = []
for name, model in eval_models.items():
    ckpt = MODELS_DIR / f'best_{name.lower()}.pt'
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"✅ Loaded {name} weights from {ckpt.name}")
    else:
        print(f"⚠️ Checkpoint for {name} not found. Using current weights.")
        
    metrics, y_true, y_pred = evaluate_model(model, data_dev, split_dev['test'], 'test')
    rows.append({'model': name, **metrics})
    
    # Plot Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, xticks_rotation='vertical', include_values=False, ax=ax, cmap='Blues'
    )
    plt.title(f'{name} Test Set Confusion Matrix (40 Classes)')
    plt.savefig(out_eval / f'confusion_matrix_{name.lower()}.png', dpi=180, bbox_inches='tight')
    plt.show()

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(out_eval / 'metrics.csv', index=False)
print("=== Final Test Benchmark Metrics ===")
display(metrics_df)

plot_model_comparison(metrics_df, out_eval / 'model_comparison.png')

---
## 8. Model Explainability, Embeddings & Homophily (Task 08)
Analyze learned 256-dimensional node representations using PCA and t-SNE projections, and quantify local neighbourhood label agreement (homophily).

In [ ]:
from src.explainability.embeddings import extract_embeddings, plot_embedding_projection
from src.explainability.neighborhood_analysis import neighbour_label_agreement

out_exp = RESULTS_DIR / 'explainability'
out_exp.mkdir(parents=True, exist_ok=True)

# Extract learned embeddings from GCN
embeddings = extract_embeddings(gcn_model, data_dev)
labels = data.y.squeeze().cpu().numpy()

print(f"Extracted hidden embeddings shape: {embeddings.shape}")

# Plot PCA & t-SNE Projections
print("Generating PCA 2D embedding projection...")
plot_embedding_projection(embeddings, labels, out_exp / 'pca_embeddings.png', method='pca')

print("Generating t-SNE 2D embedding projection (subsample)...")
plot_embedding_projection(embeddings, labels, out_exp / 'tsne_embeddings.png', method='tsne')

# Compute neighbourhood label agreement (Homophily) on sample test nodes
sample_nodes = split_dev['test'][:5].cpu().numpy()
print("\n=== Local Neighbourhood Label Agreement (Homophily) ===")
for nid in sample_nodes:
    agreement = neighbour_label_agreement(data.edge_index, data.y, int(nid))
    true_cls = int(data.y[nid])
    print(f"Node {nid:6d} (Class {true_cls:2d}) -> Local Homophily Agreement: {agreement*100:.1f}%")

---
## 9. Interactive Paper Category Inference Demo
Predict the subject category and confidence distribution for any given paper in the citation graph.

In [ ]:
# Interactive Inference Demo
def predict_paper_category(node_index: int, top_k: int = 5):
    gcn_model.eval()
    with torch.no_grad():
        out = gcn_model(data_dev.x, data_dev.edge_index)
        probs = torch.softmax(out[node_index], dim=-1).cpu().numpy()
        
    top_indices = np.argsort(probs)[::-1][:top_k]
    true_label = int(data.y[node_index])
    
    print(f"📄 Paper Node ID: {node_index}")
    print(f"🎯 Ground Truth Category: Class {true_label}")
    print(f"\nTop-{top_k} Model Predictions:")
    for rank, idx in enumerate(top_indices, 1):
        bar = "█" * int(probs[idx] * 30)
        marker = " ✅ (MATCH)" if idx == true_label else ""
        print(f"  {rank}. Class {idx:2d}: {probs[idx]*100:5.2f}% {bar}{marker}")

# Run demo on a test node
test_paper_id = int(split_dev['test'][0])
predict_paper_category(test_paper_id)